# RawFileReader integration test

This notebook exercises `RawFileAdapter` against the repository's real `sample.raw` and the bundled .NET 8 RawFileReader assemblies. It can be run directly in Google Colab or locally from either the repository root or the `tests` directory.

## Setup

The next cell installs `pythonnet` in every environment. On Google Colab, it also installs the .NET 8 runtime. Local environments must provide .NET 8 through their operating-system package manager.


In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB and shutil.which("dotnet") is None:
    subprocess.run(
        [
            "bash",
            "-c",
            "set -e; "
            "wget -q https://packages.microsoft.com/config/ubuntu/22.04/"
            "packages-microsoft-prod.deb -O /tmp/packages-microsoft-prod.deb; "
            "dpkg -i /tmp/packages-microsoft-prod.deb >/dev/null; "
            "rm /tmp/packages-microsoft-prod.deb; "
            "apt-get update -qq; "
            "apt-get install -y -qq dotnet-runtime-8.0",
        ],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pythonnet>=3.0.3"],
    check=True,
)
os.environ.setdefault("DOTNET_ROOT", "/usr/share/dotnet")

print("pythonnet installed")
print("DOTNET_ROOT:", os.environ["DOTNET_ROOT"])
print("dotnet executable:", shutil.which("dotnet") or "not found")


In [ ]:
from pathlib import Path

if IN_COLAB:
    repo_root = Path("/content/RawFileReaderPyAdapter")
    if not repo_root.is_dir():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/mzzzhunter/RawFileReaderPyAdapter.git",
                str(repo_root),
            ],
            check=True,
        )
else:
    working_directory = Path.cwd().resolve()
    repo_root = (
        working_directory.parent
        if working_directory.name == "tests"
        else working_directory
    )

if not (repo_root / "rawfilereader").is_dir():
    raise RuntimeError("Could not locate the RawFileReaderPyAdapter repository")

sys.path.insert(0, str(repo_root))

from rawfilereader import AssemblyLoadError, RawFileAdapter
from rawfilereader.loader import load_assemblies


In [ ]:
sample_raw = repo_root / "sample.raw"
assemblies = repo_root / "libs" / "Net8" / "Assemblies"
required_assemblies = (
    "OpenMcdf.dll",
    "OpenMcdf.Extensions.dll",
    "ThermoFisher.CommonCore.Data.dll",
    "ThermoFisher.CommonCore.RawFileReader.dll",
    "ThermoFisher.CommonCore.BackgroundSubtraction.dll",
)

assert sample_raw.is_file(), f"Missing integration fixture: {sample_raw}"
assert sample_raw.stat().st_size > 0, "sample.raw is empty"
for assembly_name in required_assemblies:
    assembly_path = assemblies / assembly_name
    assert assembly_path.is_file(), f"Missing assembly: {assembly_path}"

print(f"RAW fixture: {sample_raw}")
print(f"Assemblies: {assemblies}")

from tempfile import TemporaryDirectory

original_sys_path = list(sys.path)
with TemporaryDirectory() as incomplete_assemblies:
    try:
        load_assemblies(incomplete_assemblies)
    except AssemblyLoadError as error:
        assert "required DLLs were not found" in str(error)
    else:
        raise AssertionError("An incomplete assembly directory was accepted")
assert sys.path == original_sys_path
print("Incomplete assembly validation left sys.path unchanged")


In [ ]:
from rawfilereader.adapter import (
    _apply_mass_range,
    _average_centroid_peaks,
    _find_closest,
    _linear_interp,
    _normalize_to_tic,
    _py_subtract_peaks,
)

masses, intensities = _apply_mass_range(
    [99.0, 100.0, 101.0], [1.0, 2.0, 3.0], (100.0, 101.0)
)
assert masses == [100.0, 101.0]
assert intensities == [2.0, 3.0]
assert _normalize_to_tic([1.0, 3.0]) == [0.25, 0.75]
assert _find_closest(100.0, [100.0004], tol_ppm=5.0) == 0
assert _find_closest(100.0, [100.0004], tol_ppm=3.0) is None
assert _linear_interp([1.5], [1.0, 2.0], [10.0, 20.0]) == [15.0]

subtracted_masses, subtracted_intensities = _py_subtract_peaks(
    [100.0, 101.0], [10.0, 5.0], [100.0004, 101.0], [4.0, 6.0]
)
assert subtracted_masses == [100.0]
assert subtracted_intensities == [6.0]

averaged_masses, averaged_intensities = _average_centroid_peaks(
    [[100.0], [100.0002]], [[10.0], [30.0]]
)
assert abs(averaged_masses[0] - 100.00015) < 1e-10
assert averaged_intensities == [20.0]
print("Pure-Python spectral helper tests passed")


In [ ]:
adapter = RawFileAdapter(str(sample_raw), libs_dir=str(assemblies))
assert not adapter.is_open

with adapter:
    assert adapter.is_open
    first_scan, last_scan = adapter.get_scan_range()
    assert first_scan >= 1
    assert last_scan >= first_scan

    file_info = adapter.get_file_info()
    first_scan_data = adapter.get_centroid_stream(first_scan)
    averaged_scan = adapter.average_scans([first_scan])
    assert averaged_scan.first_scan == first_scan
    assert averaged_scan.last_scan == first_scan

    try:
        adapter.average_scans([])
    except ValueError as error:
        assert "at least one" in str(error)
    else:
        raise AssertionError("average_scans accepted an empty scan list")

    try:
        adapter.average_scans_in_range(first_scan + 1, first_scan)
    except ValueError as error:
        assert "less than or equal" in str(error)
    else:
        raise AssertionError("average_scans_in_range accepted reversed bounds")

    available_instrument_types = []
    for instrument_index in range(adapter.get_instrument_count()):
        device_type = adapter.get_instrument_type(instrument_index)
        if device_type not in available_instrument_types:
            available_instrument_types.append(device_type)
    assert available_instrument_types, "No selectable instruments found in sample.raw"
    target_type = available_instrument_types[-1]
    target_instance = 1
    adapter.select_instrument(target_type, target_instance)
    selected_instrument = adapter.get_instrument_data()
    assert selected_instrument.device_type == target_type
    assert selected_instrument.instance_number == target_instance

    print(f"File: {file_info.file_name}")
    print(f"Scan range: {first_scan}–{last_scan}")
    print(f"First scan centroid peaks: {len(first_scan_data.masses)}")
    print(f"Averaged scan peaks: {len(averaged_scan.masses)}")
    print(f"Selected instrument: {target_type} instance {target_instance}")

assert not adapter.is_open
print("Integration test passed; the native RAW file was closed.")
